# 14.3 Model Drift

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 14: MLOps & Monitoring

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-14-mlops-and-monitoring/14.03_model-drift.ipynb)

Companion code for Section 14.3 of the book. Run the cells in order from top to bottom.

**Example 14.3.1** · bash

```bash
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.datasets import load_breast_cancer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
import warnings; warnings.filterwarnings('ignore')

# ── Model Performance Monitor ─────────────────────────────────────
class ModelPerformanceMonitor:
    '''
    Tracks model performance over time windows.
    Detects accuracy degradation and distribution shifts.
    '''

    def __init__(self, model, alert_auc_drop: float = 0.03,
                  alert_psi_threshold: float = 0.25):
        self.model               = model
        self.alert_auc_drop      = alert_auc_drop
        self.alert_psi_threshold = alert_psi_threshold
        self.baseline_proba      = None
        self.baseline_auc        = None
        self.history             = []

    def set_baseline(self, X_ref: np.ndarray, y_ref: np.ndarray) -> None:
        '''Establish baseline from training/validation data.'''
        self.baseline_proba = self.model.predict_proba(X_ref)[:, 1]
        self.baseline_auc   = roc_auc_score(y_ref, self.baseline_proba)
        self.baseline_brier = brier_score_loss(y_ref, self.baseline_proba)
        print(f'Baseline set: AUC={self.baseline_auc:.4f}  Brier={self.baseline_brier:.4f}')

    def evaluate_window(self, X_window: np.ndarray,
                          y_window: np.ndarray,
                          window_label: str) -> Dict:
        '''Evaluate model on a production window.'''
        proba  = self.model.predict_proba(X_window)[:, 1]
        auc    = roc_auc_score(y_window, proba)
        brier  = brier_score_loss(y_window, proba)

        # KS test on prediction distribution
        ks_stat, ks_p = ks_2samp(self.baseline_proba, proba)

        # Performance delta
        auc_drop   = self.baseline_auc - auc
        brier_rise = brier - self.baseline_brier

        # Alert conditions
        auc_alert   = auc_drop   > self.alert_auc_drop
        pred_drift  = ks_p < 0.05

        result = {
            'window':      window_label,
            'auc':         round(auc, 4),
            'auc_drop':    round(auc_drop, 4),
            'brier':       round(brier, 4),
            'brier_rise':  round(brier_rise, 4),
            'pred_mean':   round(proba.mean(), 4),
            'pred_std':    round(proba.std(), 4),
            'ks_stat':     round(ks_stat, 4),
            'ks_p':        round(ks_p, 6),
            'auc_alert':   auc_alert,
            'pred_drift':  pred_drift,
            'status':      'ALERT' if (auc_alert or pred_drift) else 'OK',
        }
        self.history.append(result)
        return result

    def print_monitoring_report(self) -> None:
        if not self.history: return
        df = pd.DataFrame(self.history)
        print(f'{'Window':12s} {'AUC':>8} {'Drop':>8} {'Pred Mean':>10} {'Status':>10}')
        print('='*55)
        for _, row in df.iterrows():
            icon = ' ' if row['status']=='ALERT' else ' '
            print(f'{row["window"]:12s} {row["auc"]:>8.4f} {row["auc_drop"]:>8.4f} {row["pred_mean"]:>10.4f}
{icon}{row["status"]:>9}')


# ── Simulate performance degradation over time ────────────────────
data = load_breast_cancer()
X, y = data.data, data.target

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

model = Pipeline([('sc', StandardScaler()),
                   ('gb', GradientBoostingClassifier(n_estimators=200, random_state=42))])
model.fit(X_tr, y_tr)

monitor = ModelPerformanceMonitor(model, alert_auc_drop=0.03)
monitor.set_baseline(X_te, y_te)

print('\n=== Model Performance Monitoring Over Time Windows ===')

np.random.seed(42)
# Simulate 6 monthly windows with increasing drift
for month in range(1, 7):
    # Add increasing noise and distribution shift
    drift_strength = month * 0.15
    X_month = X_te + np.random.normal(0, drift_strength, X_te.shape)
    y_month = y_te.copy()

    # Introduce concept drift in later months
    if month >= 4:
        flip_idx = np.random.choice(len(y_month), int(len(y_month)*0.05*(month-3)), replace=False)
        y_month[flip_idx] = 1 - y_month[flip_idx]

    result = monitor.evaluate_window(X_month, y_month, f'Month-{month}')

monitor.print_monitoring_report()

# Calibration analysis
print('\n=== Calibration Check (Month 1 vs Month 6) ===')
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Model Drift Monitoring', fontsize=14, fontweight='bold')

# AUC over time
ax = axes[0]
df_hist = pd.DataFrame(monitor.history)
ax.plot(range(1, 7), df_hist['auc'], 'b-o', ms=8, lw=2.5, label='Monthly AUC')
ax.axhline(monitor.baseline_auc, color='green', ls='--', lw=2,
          label=f'Baseline AUC={monitor.baseline_auc:.4f}')
ax.axhline(monitor.baseline_auc - monitor.alert_auc_drop, color='red', ls='--', lw=2,
          label=f'Alert threshold={monitor.baseline_auc-monitor.alert_auc_drop:.4f}')
alert_months = df_hist[df_hist['auc_alert']].index + 1
ax.scatter(alert_months, df_hist[df_hist['auc_alert']]['auc'],
          s=200, c='red', marker='*', zorder=10, label='Alert triggered')
ax.set_xlabel('Month'); ax.set_ylabel('AUC')
ax.set_title('AUC Over Time\n(degradation detected in Month 4+)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Prediction distribution shift
ax = axes[1]
ax.plot(range(1, 7), df_hist['pred_mean'], 'r-o', ms=8, lw=2.5, label='Pred mean')
ax.fill_between(range(1,7),
               df_hist['pred_mean'] - df_hist['pred_std'],
               df_hist['pred_mean'] + df_hist['pred_std'],
               alpha=0.2, color='red', label='±1 std')
ax.set_xlabel('Month'); ax.set_ylabel('Mean P(y=1)')
ax.set_title('Prediction Distribution Over Time')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Brier score over time
ax = axes[2]
ax.plot(range(1, 7), df_hist['brier'], 'g-o', ms=8, lw=2.5, label='Brier Score')
ax.set_xlabel('Month'); ax.set_ylabel('Brier Score (lower=better)')
ax.set_title('Calibration (Brier Score) Over Time')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('model_drift_monitoring.png', dpi=150, bbox_inches='tight')
plt.show()
```